# Project Brazilian e-commerce by Olist

## Modeling Schema

<img src=..\docs\assets\data_schema.png width="900">


Source: https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce

## Data Dictionary

| Features | Description | Tables |
| :--- | :--- | :--- |
| customer_city | customer city name | customers |
| customer_id | key to the orders dataset. Each order has a unique customer_id. | customers, orders |
| customer_state | customer state | customers |
| customer_unique_id | unique identifier of a customer. | customers |
| customer_zip_code_prefix | first five digits of customer zip code | customers |
| freight_value | item freight value item (if an order has more than one item the freight value is splitted between items) | items |
| geolocation_city | city name | geolocation |
| geolocation_lat | latitude | geolocation |
| geolocation_lng | longitude | geolocation |
| geolocation_state | state | geolocation |
| geolocation_zip_code_prefix | first 5 digits of zip code | geolocation |
| order_approved_at | Shows the payment approval timestamp. | orders |
| order_delivered_carrier_date | Shows the order posting timestamp. When it was handled to the logistic partner. | orders |
| order_delivered_customer_date | Shows the actual order delivery date to the customer. | orders |
| order_estimated_delivery_date | Shows the estimated delivery date that was informed to customer at the purchase moment. | orders |
| order_id | order unique identifier | items, payments, reviews, orders |
| order_item_id | sequential number identifying number of items included in the same order | items |
| order_purchase_timestamp | Shows the purchase timestamp. | orders |
| order_status | Reference to the order status (delivered, shipped, etc). | orders |
| payment_installments | number of installments chosen by the customer. | payments |
| payment_sequential | a customer may pay an order with more than one payment method. If he does so, a sequence will be created to | payments |
| payment_type | method of payment chosen by the customer. | payments |
| payment_value | transaction value. | payments |
| price | item price | items |
| product_category_name | root category of product, in Portuguese. | products |
| > product_description_lenght | number of characters extracted from the product description. | products |
| product_height_cm | product height measured in centimeters. | products |
| product_id | product unique identifier | items, products |
| product_length_cm | product length measured in centimeters. | products |
| > product_name_lenght | number of characters extracted from the product name. | products |
| product_photos_qty | number of product published photos. | products |
| product_weight_g | product weight measured in grams. | products |
| product_width_cm | product width measured in centimeters. | products |
| review_answer_timestamp | Shows satisfaction survey answer timestamp. | reviews |
| review_comment_message | Comment message from the review left by the customer, in Portuguese. | reviews |
| review_comment_title | Comment title from the review left by the customer, in Portuguese. | reviews |
| review_creation_date | Shows the date in which the satisfaction survey was sent to the customer. | reviews |
| review_id | unique review identifier | reviews |
| review_score | Note ranging from 1 to 5 given by the customer on a satisfaction survey. | reviews |
| seller_city | seller city name | sellers |
| seller_id | seller unique identifier | items, sellers |
| seller_state | seller state | sellers |
| seller_zip_code_prefix | first 5 digits of seller zip code | sellers |
| shipping_limit_date | Shows the seller shipping limit date for handling the order over to the logistic partner. | items |


## Imports and configs

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from src.eda_utils import *
from src.plots import plot_numeric_distribution, plot_categorical_counts

In [ ]:
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)

In [ ]:
DATE_COLS = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
    "shipping_limit_date",
    "review_creation_date",
    "review_answer_timestamp",
]

CAT_COLS = [
    "customer_city",
    "customer_state",
    "order_status",
    "payment_types",
    "product_category_name",
    "product_category_name_english",
    "seller_city",
    "seller_state",
]

In [ ]:
pd.reset_option("display.max_rows")

## Reading Data

In [ ]:
customers = pd.read_csv("../data/raw/olist_customers_dataset.csv")
geolocation = pd.read_csv("../data/raw/olist_geolocation_dataset.csv")
order_items = pd.read_csv("../data/raw/olist_order_items_dataset.csv")
order_payments = pd.read_csv("../data/raw/olist_order_payments_dataset.csv")
order_reviews = pd.read_csv("../data/raw/olist_order_reviews_dataset.csv")
orders = pd.read_csv("../data/raw/olist_orders_dataset.csv")
products = pd.read_csv("../data/raw/olist_products_dataset.csv")
sellers = pd.read_csv("../data/raw/olist_sellers_dataset.csv")

In [ ]:
dfs = {
    "customers": customers,
    "geolocation": geolocation,
    "items": order_items,
    "payments": order_payments,
    "reviews": order_reviews,
    "orders": orders,
    "products": products,
    "sellers": sellers,
}

## Initial EDA

### Summary Statistics

In [ ]:
for name, df in dfs.items():
    print(f"{name.upper()} {df.shape}:")
    display(dataframe_summary(df))

### Observations

- O order_id está duplicado em orders (deveria ser único)? `Não`
- review_id duplicado ou múltiplos review_id para o mesmo order_id em order_reviews? `Sim, será tratado no merge dos dados` 
- Unicidade de order_id + order_item_id em order_items? `Sim, não há duplicidade das chaves compostas`
- product_id duplicado em products, seller_id duplicado em sellers? `Não`
- review_score sempre entre 1 e 5 em order_reviews? `Sim`
- Valores negativos/zero em price, freight_value (order_items), payment_value (order_payments), product_weight_g (products)? `Não há valores negativos, porém as colunas freight_value, payment_installments, payment_value e product_weight_g estão com valor mínimo zero, o que pode fazer sentido no caso de freight_value, mas não para os demais.`

In [ ]:
dfs["payments"].payment_installments.unique()

In [ ]:
# Verifica se a combinação order_id + order_item_id é única
duplicates = dfs["items"].duplicated(subset=["order_id", "order_item_id"]).sum()
print(f"Linhas duplicadas na chave composta: {duplicates}")

In [ ]:
strange_values = {
    "items": ["freight_value"],
    "payments": ["payment_installments", "payment_value"],
    "products": ["product_weight_g"],
}

for table, columns in strange_values.items():
    for col in columns:
        mask = dfs[table][col] == 0
        print(f"coluna {col} zerada")
        display(dfs[table][mask])
        if table == "items":
            display(dfs[table][mask].value_counts("seller_id"))
            # display(show_multi_item_orders(dfs[table][mask], required_cols = dfs[table].columns))

- Frete zerado pode ser considerado valor legitimo e por se tratar de vendedores especificos demonstra um padrão, vale a pena investigar mais sobre após a junção das tabelas e checar se há algum padrão associado a categoria, localização ou preço.
- payment_installments zerado é provavelmente algum erro, porém tratando-se apenas de dois pedidos tem impacto mínimo no projeto.
- payment_value zerado pode representar erro, principalmente para payment_type not_defined vale a pena verificar novamente após o merge.
- product_weight_g zerado é um valor impossivel, o que também indica erro.

O tratamento desses erros e dados nulos serão feitos após o merge das tabelas, pois com mais contexto é possivel tomar uma decisão mais acertiva com relação a descartar ou preencher os valores.

In [ ]:
mask = dfs["orders"]["order_delivered_customer_date"].isna()
dfs["orders"][mask].value_counts("order_status")

- A feature order_delivered_customer_date é essencial para o objetivo de prever possivel atraso e status delivered é o que faz mais sentido neste contexto, então ter apenas 8 nulos nesta categoria é um bom sinal. 

### Cardinality Checking

1. order_items — esperado 1:N

In [ ]:
# Conta quantos itens (linhas) cada order_id tem
items_per_order = dfs["items"].groupby("order_id").size()

# Distribuição: quantos pedidos têm 1 item, 2 itens, 3 itens, etc.
print(items_per_order.value_counts().sort_index())

# Estatística resumida
print(items_per_order.describe())

2. order_payments — checar se existe 1:N

In [ ]:
# Conta quantas linhas de pagamento cada order_id tem
payments_per_order = dfs["payments"].groupby("order_id").size()

print(payments_per_order.value_counts().sort_index())
print(payments_per_order.describe())

# Opcional: olhar exemplos de pedidos com múltiplas linhas de pagamento
multi_payment_orders = payments_per_order[payments_per_order > 1].index
dfs["payments"][dfs["payments"]["order_id"].isin(multi_payment_orders)].sort_values(
    "order_id"
).head(20)

3. order_reviews — checar se é ~1:1

In [ ]:
reviews_per_order = dfs["reviews"].groupby("order_id")["review_id"].nunique()

multi_review_orders = (reviews_per_order > 1).sum()
print(
    f"Pedidos com múltiplos reviews: {multi_review_orders} ({multi_review_orders / reviews_per_order.shape[0]:.2%})"
)

multi_review_orders = reviews_per_order[reviews_per_order > 1].index
df_multi_review = dfs["reviews"][
    dfs["reviews"]["order_id"].isin(multi_review_orders)
].sort_values("order_id")

display(
    df_multi_review[
        [
            "order_id",
            "review_id",
            "review_score",
            "review_creation_date",
            "review_answer_timestamp",
        ]
    ].head(20)
)

- Como o usuario pode ter atualizada a sua review, a estrategia que faz mais sentido é manter apenas o ultimo review_answer_timestamp.

Para dar prosseguimento as analises irei juntar todas as tabelas exceto geolocation (por enquanto), no entato surgem algumas perguntas que devem ser respondidas antes do merge: 
- `"O que representa uma linha no meu dataset unificado?"` ->  após breve analise cheguei a conclusão que o nivel mais granular que ainda faz sentido é o de item do pedido (order_items), pois produto, vendedor, frete e preço variam por item e não por um pedido inteiro.
- `"Como as demais tabelas se relacionam com o item do pedido?"` -> Isso pode ser respondido pela relação de cardinalidade:

| Tabela | Relação com "item do pedido" |
|---|---|
| `orders` | 1 pedido → N itens (então 1:m em relação ao grão) |
| `order_items` | **é o próprio grão** |
| `products` | N itens podem repetir o mesmo produto (m:1) |
| `sellers` | N itens podem repetir o mesmo vendedor (m:1) |
| `customers` | 1 pedido = 1 cliente, mas N itens do mesmo pedido repetem o cliente (m:1 em relação ao grão) |
| `payments` | 1 pedido pode ter N pagamentos — **não bate direto com o grão "item"** |
| `reviews` | 1 pedido normalmente tem 1 review, raramente mais — **também não bate diretamente** |

payments e reviews são relações "por pedido", não "por item". Se eu simplesmente desse merge direto delas contra o dataset já expandido por item, qualquer pedido com 2 itens e 2 pagamentos viraria 4 linhas (2×2) — um produto cartesiano que não representa nada real, só um artefato do merge.

Dito isto, a ordem do merge fica:

```
orders → customers → order_items → products → sellers → payments (agregado) → reviews (dedup)
```

- **`orders + customers` primeiro**: relação 1:1 real (cada pedido tem exatamente um cliente), então é seguro e não altera a contagem de linhas. Fiz isso cedo porque é "barato" e não introduz risco.
- **`+ order_items`**: aqui, e só aqui, o dataset "explode" de fato — de propósito, porque é essa a granularidade que eu quero.
- **`+ products` e `+ sellers`**: ambos são m:1 em relação ao grão atual (vários itens podem apontar pro mesmo produto/vendedor), então o merge não altera a contagem de linhas — só adiciona colunas.
- **`+ payments`**: como *não* é m:1 em relação ao grão, eu não faço merge direto. Primeiro **agrego** (`groupby('order_id')`) para colapsar a granularidade de payments para "por pedido", e só depois faço o merge — agora sim é m:1 seguro.
- **`+ reviews`**: mesmo raciocínio — dedupliquei por `order_id` antes, transformando numa relação segura.



In [ ]:
# Roda o script de merge das tabela, e trata o problemas encontrados anteriormente
%run ../src/data_processing.py

## Integrity Checks

In [ ]:
df_processed = pd.read_parquet("../data/processed/olist_processed.parquet")
df_processed.head(3)

### Summary Statistics

In [ ]:
# save_table_as_image(
#     dataframe_summary(df_processed), "../reports/figures/table_preview.png"
# )

In [ ]:
dataframe_summary(df_processed)

In [ ]:
df_processed.n_payment_methods.unique()

### Observations

1. **`shipping_limit_date` com máximo em 2020-04-09.** O dataset cobre 2016-2018 (compras até 2018-09), então essa data está fora do período. Vale investigar as linhas afetadas.

2. **`product_weight_g` com mínimo 0.0.** Como já visto antes peso zero é impossível, então irei inserir nulo.

3. **`payment_installments_max` com mínimo 0.** Um pagamento com zero parcelas é suspeito.

4. **112.650 linhas para 98.666 pedidos.** É o grão de item, como esperado. O `order_id` mais frequente aparece 21 vezes e `order_item_id` chega a 21, então há um pedido com 21 itens, que pesa 21 vezes mais que um pedido comum se colapsar para o nível de pedido.

5. **A tabela `orders` original tem 99.441 pedidos.** Portanto o `inner` em `merge_order_items` descartou cerca de 775 pedidos sem itens. O `how="inner"`, fez a categoria `created` de `order_status` desaparecer e maior partes dos pedidos `unavailable` também não foram encontrados.

6. **`payment_value_total` está inflado.** A média é 180,28, contra 140,64 de `price + freight_value` (120,65 + 19,99). Em pedidos com vários itens, cada linha repete o pagamento total do pedido, ou seja, essa coluna no grão de item não deve ser somada.

7. **`customer_id` e `order_id` têm 98.666 únicos cada**, e `customer_unique_id` tem 95.420. Ou seja, há pouca recompra, algo que pode aparecer na análise de negócio.

8. `order_delivered_customer_date` tem 2.454 nulos, quase igual às 2.453 linhas que não estão como `delivered` (112.650 − 110.197). É necessario cruzar com `order_status` para confirmar que os nulos coincidem com pedidos não entregues.

9. `order_delivered_carrier_date` (1.194) e `order_approved_at` (15) seguem a mesma lógica.

10. Colunas `review_*` com 942 nulos são pedidos sem avaliação.

11. `review_comment_title` (88%) e `review_comment_message` (58%) são campos opcionais. Provavelmente o mais útil é uma flag como `has_comment`.

**Nos produtos:**
- 1.603 nulos idênticos em `product_category_name`, `product_name_lenght`, `product_description_lenght` e `product_photos_qty`, o que sugere as mesmas linhas com metadados ausentes.
- `product_category_name_english` tem 1.627 nulos, ou seja, **24 a mais** que a categoria em português. São categorias que não estão na tabela de tradução, e dá para traduzir manualmente.
- Só 3 linhas sem pagamento (provavelmente um único pedido com 3 itens).

#### Distribuições

- **Assimetria forte** em `price` (skew 7,9 e curtose 120, máximo 6.735 contra mediana de 74,99), em `payment_value_total` (skew 13,9) e em `product_weight_g`. Para o XGBoost isso não exige transformação, mas para gráficos e modelos lineares a escala logarítmica ajuda.
- **`review_score` concentrado em 5**: 56,5% das linhas (63.157 de 111.708), com skew de −1,27. Se o alvo for "avaliação ruim", a classe positiva será minoritária.
- **97,8% dos pedidos estão como `delivered`.** Para o modelo de atraso, só esses têm data de entrega para calcular o alvo.
- `freight_value` com mínimo 0.0 pode ser legítimo (frete grátis), diferente do peso.

#### Detalhes menores

- `product_name_lenght` e `product_description_lenght` mantêm o erro de grafia do dataset original (`lenght`). Como você usa nomes em inglês, vale renomear para `length`.
- `review_score`, `n_payment_methods` e `payment_installments_max` estão como `float64` só por causa dos nulos. O `Int64` do pandas resolveria, mas não é urgente.

### outros
- `order_id` em `order_items`/`order_payments`/`order_reviews` que não bateram com `orders` (ficaram nulos após o `left join`)?Não surgiram valores nulos para `order_id`
- `product_id`/`seller_id` em `order_items` que não bateram com `products`/`sellers`?
- `customer_id` em `orders` que não bateu com `customers`?
- Consitência temporal `order_purchase_timestamp` ≤ `order_approved_at` ≤ `order_delivered_carrier_date` ≤ `order_delivered_customer_date` ≤ `review_creation_date`?

**Nulos que só existem/fazem sentido pós-merge**
- Nulos em `review_score` cruzando com `order_status` (já é a tabela unificada) — para decidir estratégia de exclusão do Model 2
- Nulos em `order_delivered_customer_date` cruzando com `order_status` na base final, já no grain de pedido — para o Model 1

**Validação de agregação**
- Conferir que `payment_value_total` (após `groupby("first")`) bate com a soma esperada por pedido, sem duplicação por item
- Conferir que `n_items`, `n_distinct_sellers`, `avg_product_weight` etc. batem com o esperado por amostragem manual de alguns `order_id`

### Dates

In [ ]:
df_processed.sort_values(by=["shipping_limit_date"], ascending=False)[DATE_COLS]

In [ ]:
for col in DATE_COLS:
    mask = df_processed[col].dt.year >= 2019
    print(f'{col} - {len(df_processed[mask])}')

In [ ]:
mask = df_processed['shipping_limit_date'].dt.year >= 2019
df_processed[mask]

In [ ]:
df_processed.loc[mask, "shipping_limit_date"] = pd.NaT

### order_delivered_customer_date

In [ ]:
mask = df_processed.order_status != 'delivered'
mask.sum()

In [ ]:
df_processed.order_status.value_counts()

In [ ]:
df_processed.loc[mask, DATE_COLS].isna().sum() # order_status != 'delivered'

In [ ]:
df_processed.loc[~mask, DATE_COLS].isna().sum() # order_statu == 'delivered'

- Quase todos da coluna `order_delivered_customer_date` que não estavam com o order_status como "delivered" estão com valor nulo na coluna order_delivered_customer_date, o que é esperado visto que não tem como um pedido ter data de entrega sem ter sido entregue. Em contrapartida há apenas 8 items sem data de entrega com o order_status como "delivered". De forma similar a coluna `order_delivered_carrier_date` tem 2 items sem data.
- Como um dos objetivos é prever se o pedido ira atrasar, o recorte ideal parece ser os pedidos da categoria order_status delivered.  

### Consitência temporal

In [ ]:
DATE_ORDER_RULES = [
    ("order_purchase_timestamp", "order_approved_at"),
    ("order_approved_at", "order_delivered_carrier_date"),
    ("order_delivered_carrier_date", "order_delivered_customer_date"),
    ("order_purchase_timestamp", "order_delivered_customer_date"),
    ("order_purchase_timestamp", "order_estimated_delivery_date"),
    ("order_approved_at", "order_delivered_customer_date"),
]


def check_date_integrity(df: pd.DataFrame) -> pd.DataFrame:
    """Verifica se as datas do pedido seguem a ordem cronológica esperada.

    Retorna um resumo com uma linha por regra, contendo quantas linhas
    foram checadas e quantas violaram a ordem.
    """
    results = []

    for earlier, later in DATE_ORDER_RULES:
        # Só compara linhas em que as duas datas existem (nulos estruturais são ignorados)
        both_present = df[earlier].notna() & df[later].notna()
        violations = both_present & (df[later] < df[earlier])

        checked_rows = int(both_present.sum())
        violation_count = int(violations.sum())

        results.append(
            {
                "rule": f"{earlier} <= {later}",
                "checked_rows": checked_rows,
                "violations": violation_count,
                "violation_pct": round(100 * violation_count / max(checked_rows, 1), 2),
            }
        )

    return pd.DataFrame(results)

In [ ]:
summary = check_date_integrity(df_processed)
display(summary)

In [ ]:
mask = (
    (df_processed["order_approved_at"].notna() & df_processed["order_delivered_carrier_date"].notna()) & (df_processed["order_approved_at"] >= df_processed["order_delivered_carrier_date"])
)
df_processed[mask]

In [ ]:
# Isola as linhas inconsistentes e calcula a magnitude da violação
mask = df_processed["order_approved_at"] > df_processed["order_delivered_customer_date"]
violations = df_processed.loc[mask].copy()
violations["gap_hours"] = (
    violations["order_approved_at"] - violations["order_delivered_customer_date"]
).dt.total_seconds() / 3600

display(violations["gap_hours"].describe())
display(violations["order_status"].value_counts())

### Others

In [ ]:
mask = df_processed["product_weight_g"] <= 0
df_processed.loc[mask]

In [ ]:
df_processed.loc[mask, "product_weight_g"] = pd.NA

In [ ]:
(df_processed.groupby(["order_id"])["order_item_id"].value_counts() > 1).sum()

Isso significa que não há items duplicados

### Payments

In [ ]:
mask = df_processed["payment_installments_max"] <= 0
df_processed[mask]

In [ ]:
df_processed[mask]["order_id"].value_counts()

In [ ]:
df_processed.loc[mask, "payment_installments_max"] = pd.NA

Há apenas um cliente sem informações de pagamento registradas, provavelmente durante o merge com a tabela payments não foi encontrado o order_id

In [ ]:
multi_item_df = show_multi_item_orders(df_processed)
multi_item_df

In [ ]:
consistency_df = check_payment_consistency(df_processed)

# resumo rápido
print(consistency_df["is_consistent"].value_counts())
print(f"Total de pedidos inconsistentes: {(~consistency_df['is_consistent']).sum()}")

# ver só os inconsistentes, ordenados pela maior diferença
inconsistent = consistency_df[~consistency_df["is_consistent"]].sort_values(
    "difference", ascending=False
)
inconsistent

In [ ]:
inconsistent.payment_types.value_counts()

Apenas uma pequena quantidade de pagamentos apresenta inconsistencia, o que significa que o merge com a tabela payments foi bem sucedido e os valores discrepantes podem ocorrer por sujeira ou outros fatores.  

### Order Status

In [ ]:
plot_categorical_counts(df_processed, "order_status", multicolor=True)

In [ ]:
# Pedidos sem data de entrega
customer_date_null = df_processed["order_delivered_customer_date"].isna()
plot_categorical_counts(
    df_processed[customer_date_null], "order_status", multicolor=True
)

# Inserção e tratamento

In [ ]:
mask = df_processed["payment_installments_max"] <= 0
display(df_processed[mask])
mask = df_processed["product_weight_g"] <= 0
display(df_processed[mask])

In [ ]:
# Cria coluna com pedidos atrasados preservando os valores nulos
df_processed["delayed"] = (
    df_processed["order_delivered_customer_date"].dt.normalize()
    > df_processed["order_estimated_delivery_date"].dt.normalize()
).mask(df_processed["order_delivered_customer_date"].isna())
df_processed["delayed"].sum()

In [ ]:
plot_categorical_counts(
    df_processed[df_processed["delayed"] == True], "order_status", multicolor=True
)

In [ ]:
plot_categorical_counts(df_processed, "review_score", multicolor=True)

In [ ]:
df_processed["payment_types"].value_counts(dropna=False)

In [ ]:
df_processed.groupby("order_id")["review_id"].count().sort_values(ascending=True)

In [ ]:
verify_items = show_multi_item_orders(
    df_processed,
    required_cols=[
        "order_id",
        "product_id",
        "product_category_name",
        "product_weight_g",
        "price",
        "freight_value",
        "payment_value_total",
        "review_id",
        "review_comment_message",
        "review_score",
        "review_creation_date",
        "seller_id",
        "seller_city",
        "order_status",
        "delayed",
    ],
)

verify_items.head(40)